# RAG with LangChain

This notebook rebuilds the manual RAG pipeline implemented in the
previous chapter using LangChain abstractions.

The objective is to understand how LangChain represents and simplifies
the main components of a RAG pipeline:

1. Documents
2. Embeddings
3. Vector storage
4. Retrieval
5. Prompt construction
6. LLM generation

In [ ]:
!pip install -q \
    "google-auth==2.49.0" \
    langchain \
    langchain-huggingface \
    langchain-google-genai \
    sentence-transformers

In [ ]:
!pip install -q jedi

In [ ]:
!pip check

In [ ]:
from google.colab import userdata

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "gemini-3.5-flash-lite"

In [ ]:
texts = [
    """
    Amazon S3 is an object storage service designed to store and retrieve
    large amounts of data. Data in S3 is stored as objects inside buckets.
    """,
    """
    Amazon EC2 provides resizable virtual computing capacity in the cloud.
    Users can launch virtual machines called instances.
    """,
    """
    AWS Lambda is a serverless compute service that runs code in response
    to events. It does not require users to provision or manage servers.
    """,
    """
    Amazon RDS is a managed relational database service. It supports
    relational database engines such as PostgreSQL, MySQL and MariaDB.
    """,
    """
    Amazon CloudFront is a content delivery network. It distributes
    content through edge locations to reduce latency.
    """
]

## 1. Abstraction Level 1 - Vector Store Similarity Search

Documents
    -> HuggingFaceEmbeddings
    -> InMemoryVectorStore
    -> similarity_search()

In [ ]:
documents = [
    Document(page_content=text.strip())
    for text in texts
]

In [ ]:
print(documents[0])

page_content='Amazon S3 is an object storage service designed to store and retrieve
    large amounts of data. Data in S3 is stored as objects inside buckets.'


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={
        "normalize_embeddings": True
    }
)

In [ ]:
test_embedding = embeddings.embed_query(
    "Which AWS service stores files?"
)

print(type(test_embedding))
print(len(test_embedding))

<class 'list'>
384


In [32]:
vector_store = InMemoryVectorStore(embeddings)

_ = vector_store.add_documents(documents)

In [33]:
question = "Which AWS service should I use to store files?"

results = vector_store.similarity_search(
    question,
    k=2
)

for doc in results:
    print(doc.page_content)
    print()

Amazon S3 is an object storage service designed to store and retrieve
    large amounts of data. Data in S3 is stored as objects inside buckets.

Amazon RDS is a managed relational database service. It supports
    relational database engines such as PostgreSQL, MySQL and MariaDB.



## 2. Abstraction Level 2 - Retriever

Vector Store
    -> as_retriever()
    -> retriever.invoke()

In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 2}
)

In [ ]:
retriever.invoke(question)

[Document(id='a7e93ae6-264f-4781-9051-daaf1b7515a7', metadata={}, page_content='Amazon S3 is an object storage service designed to store and retrieve\n    large amounts of data. Data in S3 is stored as objects inside buckets.'),
 Document(id='ee71c63b-078c-4e33-8b64-b9db32f4503f', metadata={}, page_content='Amazon RDS is a managed relational database service. It supports\n    relational database engines such as PostgreSQL, MySQL and MariaDB.')]

### From similarity search to a retriever

In the manual implementation, retrieval required:

query encoding -> similarity calculation -> ranking -> top-k selection

`similarity_search()` abstracts these operations behind the vector store.

`as_retriever()` adds another abstraction layer by exposing retrieval
through a common Retriever interface.

## 3. Complete RAG Chain

The following sections first orchestrate the LangChain components explicitly
and then compose them into a single runnable RAG chain.

### 3.1 Explicit orchestration with LangChain components

retriever.invoke()

format_docs()

prompt.invoke()

llm.invoke()

In [ ]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [ ]:
retrieved_docs = retriever.invoke(question)

context = format_docs(retrieved_docs)

print(context)

Amazon S3 is an object storage service designed to store and retrieve
    large amounts of data. Data in S3 is stored as objects inside buckets.

Amazon RDS is a managed relational database service. It supports
    relational database engines such as PostgreSQL, MySQL and MariaDB.


In [ ]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Answer the user's question using only the provided context.

        If the answer cannot be found in the context, say:
        "The information is not available in the provided context."

        Context:
        {context}
        """
    ),
    (
        "human",
        "{question}"
    )
])

In [ ]:
test_prompt = prompt.invoke({
    "context": context,
    "question": question
})

print(test_prompt)

messages=[SystemMessage(content='\n        Answer the user\'s question using only the provided context.\n\n        If the answer cannot be found in the context, say:\n        "The information is not available in the provided context."\n\n        Context:\n        Amazon S3 is an object storage service designed to store and retrieve\n    large amounts of data. Data in S3 is stored as objects inside buckets.\n\nAmazon RDS is a managed relational database service. It supports\n    relational database engines such as PostgreSQL, MySQL and MariaDB.\n        ', additional_kwargs={}, response_metadata={}), HumanMessage(content='Which AWS service should I use to store files?', additional_kwargs={}, response_metadata={})]


In [ ]:
api_key = userdata.get("GEMINI_API_KEY")

In [ ]:
llm = ChatGoogleGenerativeAI(
    model=LLM_MODEL,
    api_key=api_key,
)

#### Temperature is intentionally omitted because the selected model does not support this generation parameter


In [ ]:
response = llm.invoke(
    "Answer in one sentence: what is an API?"
)

print(response.text)

An API, or Application Programming Interface, is a set of rules and protocols that allows different software applications to communicate and exchange data with each other.


In [ ]:
retrieved_docs = retriever.invoke(question)

context = format_docs(retrieved_docs)

formatted_prompt = prompt.invoke({
    "context": context,
    "question": question
})

response = llm.invoke(formatted_prompt)

print(response.text)

Based on the provided context, you should use Amazon S3 to store files (as it is an object storage service designed to store and retrieve large amounts of data, with data stored as objects inside buckets).


### 3.2 Composed RAG chain

rag_chain = (...)
rag_chain.invoke(...)

In [ ]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
answer = rag_chain.invoke(
    "Which AWS service should I use to store files?"
)

print(answer)

Based on the provided context, you should use Amazon S3, which is an object storage service designed to store and retrieve large amounts of data.


In [ ]:
answer_no_context = rag_chain.invoke(
    "Which AWS service should I use to run Kubernetes clusters?"
)

print(answer_no_context)

The information is not available in the provided context.


## Key Takeaways

- LangChain provides progressively higher-level abstractions over the RAG pipeline.
- `similarity_search()` abstracts embedding-based ranking and top-k retrieval.
- A Retriever provides a common interface for retrieving relevant documents.
- Prompt templates and LLM wrappers simplify context augmentation and generation.
- LangChain Runnables allow the complete RAG workflow to be composed into a single chain.
- `rag_chain.invoke()` still performs the same fundamental RAG steps implemented manually in the previous chapter.